# 07 - Análise da Camada Gold

Este notebook é exclusivamente analítico. Ele consome dimensões, fatos e agregados já produzidos pelo pipeline, sem alterar a estrutura da camada Gold. As análises cobrem vendas, categorias, recorrência de clientes e desempenho logístico.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "datalake_mvp"
GOLD = "mvp_gold"

def gold_table(nome_tabela):
    return spark.table(f"{CATALOG}.{GOLD}.{nome_tabela}")

dim_customer = gold_table("dim_customer")
dim_seller = gold_table("dim_seller")
dim_product = gold_table("dim_product")
dim_date = gold_table("dim_date")
dim_order = gold_table("dim_order")

fact_order_items = gold_table("fact_order_items")
fact_payments = gold_table("fact_payments")

print("✅ Camada Gold carregada com sucesso.")

# Agregados materializados pelo pipeline
agg_vendas_mensais_gold = gold_table("agg_vendas_mensais")
agg_vendas_categoria_gold = gold_table("agg_vendas_categoria")
agg_clientes_gold_ref = gold_table("agg_clientes")
agg_logistica_uf_gold = gold_table("agg_logistica_uf")


In [ ]:
kpis_pedidos = (
    fact_order_items
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.count("*").alias("total_itens"),
        F.round(F.sum("price"), 2).alias("valor_produtos"),
        F.round(F.sum("freight_value"), 2).alias("valor_frete"),
        F.round(F.sum("item_total_value"), 2).alias("valor_itens_com_frete"),
        F.round(F.avg("delivery_time_days"), 2).alias("tempo_medio_entrega_dias"),
        F.sum("flag_late_delivery").alias("itens_pedidos_atrasados")
    )
)

display(kpis_pedidos)

In [ ]:
kpis_pagamentos = (
    fact_payments
    .agg(
        F.countDistinct("order_id").alias("pedidos_com_pagamento"),
        F.round(F.sum("payment_value"), 2).alias("valor_total_pagamentos")
    )
)

display(kpis_pagamentos)

In [ ]:
ticket_medio = (
    fact_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("valor_pedido")
    )
    .agg(
        F.round(F.avg("valor_pedido"), 2).alias("ticket_medio")
    )
)

display(ticket_medio)

## 6.2 - Evolução Mensal das Vendas

Nesta etapa é analisada a evolução temporal dos pedidos, itens vendidos e valores movimentados.

A análise utiliza a data de compra do pedido e preserva a granularidade da fato de itens. O número de pedidos é calculado por meio da contagem distinta de `order_id`, evitando duplicidade em pedidos que possuem mais de um item.

In [ ]:
vendas_mensais = (
    fact_order_items.alias("f")
    
    .join(
        dim_date
        .select(
            "date_key",
            "date",
            "year",
            "month"
        )
        .alias("d"),
        F.col("f.date_key") == F.col("d.date_key"),
        "inner"
    )
    
    .withColumn(
        "ano_mes",
        F.date_format(F.col("d.date"), "yyyy-MM")
    )
    
    .groupBy("ano_mes")
    
    .agg(
        F.countDistinct("f.order_id").alias("pedidos"),
        F.count("*").alias("itens_vendidos"),
        F.round(F.sum("f.price"), 2).alias("valor_produtos"),
        F.round(F.sum("f.freight_value"), 2).alias("valor_frete"),
        F.round(F.sum("f.item_total_value"), 2).alias("valor_total")
    )
    
    .orderBy("ano_mes")
)

display(vendas_mensais)

In [ ]:
vendas_mensais = (
    vendas_mensais
    .withColumn(
        "ticket_medio_produtos",
        F.round(
            F.col("valor_produtos") / F.col("pedidos"),
            2
        )
    )
    .withColumn(
        "itens_por_pedido",
        F.round(
            F.col("itens_vendidos") / F.col("pedidos"),
            2
        )
    )
)

display(vendas_mensais)

In [ ]:
# ============================================================
# Evolução mensal das vendas
# ============================================================

window_mes = Window.orderBy("ano_mes")

vendas_mensais = (
    vendas_mensais

    # Valor do mês anterior
    .withColumn(
        "valor_mes_anterior",
        F.lag("valor_total").over(window_mes)
    )

    # Crescimento percentual em relação ao mês anterior
    .withColumn(
        "crescimento_mensal_pct",
        F.when(
            F.col("valor_mes_anterior").isNotNull() &
            (F.col("valor_mes_anterior") != 0),

            F.round(
                (
                    (F.col("valor_total") - F.col("valor_mes_anterior"))
                    / F.col("valor_mes_anterior")
                ) * 100,
                2
            )
        )
    )
)

display(vendas_mensais)

In [ ]:
# ============================================================
# Crescimento mensal considerando o mês calendário anterior
# ============================================================

vendas_base = (
    vendas_mensais
    .drop("valor_mes_anterior", "crescimento_mensal_pct")
    .withColumn(
        "mes_ref",
        F.to_date(
            F.concat(F.col("ano_mes"), F.lit("-01"))
        )
    )
)

mes_anterior = (
    vendas_base
    .select(
        F.add_months(F.col("mes_ref"), 1).alias("mes_ref"),
        F.col("valor_total").alias("valor_mes_anterior")
    )
)

vendas_mensais = (
    vendas_base.alias("atual")
    .join(
        mes_anterior.alias("anterior"),
        on="mes_ref",
        how="left"
    )
    .withColumn(
        "crescimento_mensal_pct",
        F.when(
            F.col("valor_mes_anterior").isNotNull() &
            (F.col("valor_mes_anterior") != 0),
            F.round(
                (
                    (F.col("valor_total") - F.col("valor_mes_anterior"))
                    / F.col("valor_mes_anterior")
                ) * 100,
                2
            )
        )
    )
    .orderBy("mes_ref")
)

display(
    vendas_mensais.select(
        "ano_mes",
        "pedidos",
        "itens_vendidos",
        "valor_total",
        "ticket_medio_produtos",
        "itens_por_pedido",
        "valor_mes_anterior",
        "crescimento_mensal_pct"
    )
)

In [ ]:
# ============================================================
# Análise de evolução mensal
# ============================================================

evolucao_mensal = (
    vendas_mensais
    .select(
        "mes_ref",
        "ano_mes",
        "pedidos",
        "itens_vendidos",
        "valor_produtos",
        "valor_frete",
        "valor_total",
        "ticket_medio_produtos",
        "itens_por_pedido",
        "valor_mes_anterior",
        "crescimento_mensal_pct"
    )
)

# Base do mês anterior
base_mes_anterior = (
    evolucao_mensal
    .select(
        F.add_months(F.col("mes_ref"), 1).alias("mes_ref"),
        F.col("pedidos").alias("pedidos_mes_anterior"),
        F.col("ticket_medio_produtos").alias("ticket_mes_anterior")
    )
)

evolucao_mensal = (
    evolucao_mensal.alias("a")
    .join(
        base_mes_anterior.alias("b"),
        on="mes_ref",
        how="left"
    )

    # Crescimento de pedidos
    .withColumn(
        "crescimento_pedidos_pct",
        F.when(
            F.col("pedidos_mes_anterior").isNotNull() &
            (F.col("pedidos_mes_anterior") != 0),
            F.round(
                (
                    (F.col("pedidos") - F.col("pedidos_mes_anterior"))
                    / F.col("pedidos_mes_anterior")
                ) * 100,
                2
            )
        )
    )

    # Crescimento do ticket médio
    .withColumn(
        "crescimento_ticket_pct",
        F.when(
            F.col("ticket_mes_anterior").isNotNull() &
            (F.col("ticket_mes_anterior") != 0),
            F.round(
                (
                    (F.col("ticket_medio_produtos") - F.col("ticket_mes_anterior"))
                    / F.col("ticket_mes_anterior")
                ) * 100,
                2
            )
        )
    )

    .orderBy("mes_ref")
)

display(
    evolucao_mensal.select(
        "ano_mes",
        "pedidos",
        "crescimento_pedidos_pct",
        "ticket_medio_produtos",
        "crescimento_ticket_pct",
        "valor_total",
        "crescimento_mensal_pct"
    )
)

In [ ]:
# ============================================================
# Tabela analítica mensal para consumo
# ============================================================

agg_vendas_mensais = (
    evolucao_mensal
    .select(
        "mes_ref",
        "ano_mes",
        "pedidos",
        "itens_vendidos",
        "valor_produtos",
        "valor_frete",
        "valor_total",
        "ticket_medio_produtos",
        "itens_por_pedido",
        "crescimento_pedidos_pct",
        "crescimento_ticket_pct",
        "crescimento_mensal_pct"
    )
    .orderBy("mes_ref")
)

display(agg_vendas_mensais)

In [ ]:
# ============================================================
# Inspeção da dimensão de produtos
# ============================================================

dim_product.printSchema()

In [ ]:
display(
    dim_product.limit(10)
)

In [ ]:
# ============================================================
# Agregação de vendas por categoria de produto
# ============================================================

vendas_categoria = (
    fact_order_items.alias("f")
    
    .join(
        dim_product.alias("p"),
        F.col("f.product_id") == F.col("p.product_id"),
        "left"
    )
    
    .groupBy(
        F.col("p.product_category_name").alias("categoria")
    )
    
    .agg(
        F.countDistinct("f.order_id").alias("pedidos"),
        
        F.count("*").alias("itens_vendidos"),
        
        F.round(
            F.sum("f.price"),
            2
        ).alias("valor_produtos"),
        
        F.round(
            F.sum("f.freight_value"),
            2
        ).alias("valor_frete"),
        
        F.round(
            F.sum("f.item_total_value"),
            2
        ).alias("valor_total")
    )
    
    .withColumn(
        "ticket_medio_produtos",
        F.round(
            F.col("valor_produtos") / F.col("pedidos"),
            2
        )
    )
    
    .withColumn(
        "preco_medio_item",
        F.round(
            F.col("valor_produtos") / F.col("itens_vendidos"),
            2
        )
    )
    
    .withColumn(
        "frete_medio",
        F.round(
            F.col("valor_frete") / F.col("itens_vendidos"),
            2
        )
    )
    
    .orderBy(F.desc("valor_total"))
)

display(vendas_categoria)

In [ ]:
# ============================================================
# Validação de integridade produto x categoria
# ============================================================

produtos_sem_dimensao = (
    fact_order_items.alias("f")
    .join(
        dim_product.select("product_id").distinct().alias("p"),
        F.col("f.product_id") == F.col("p.product_id"),
        "left_anti"
    )
    .count()
)

print(f"Produtos sem correspondência na dimensão: {produtos_sem_dimensao:,}")

In [ ]:
categorias_nulas = (
    fact_order_items.alias("f")
    .join(
        dim_product.alias("p"),
        F.col("f.product_id") == F.col("p.product_id"),
        "left"
    )
    .filter(F.col("p.product_category_name").isNull())
    .count()
)

print(f"Itens sem categoria definida: {categorias_nulas:,}")

In [ ]:
# ============================================================
# Tratamento de categorias nulas
# + participação na receita
# ============================================================

vendas_categoria = (
    vendas_categoria
    .withColumn(
        "categoria",
        F.coalesce(
            F.col("categoria"),
            F.lit("sem_categoria")
        )
    )
)

# Receita total da base
receita_total = (
    vendas_categoria
    .agg(F.sum("valor_total").alias("receita_total"))
    .first()["receita_total"]
)

# Participação de cada categoria na receita
vendas_categoria = (
    vendas_categoria
    .withColumn(
        "participacao_receita_pct",
        F.round(
            (F.col("valor_total") / F.lit(receita_total)) * 100,
            2
        )
    )
    .orderBy(F.desc("valor_total"))
)

display(vendas_categoria)

In [ ]:
# ============================================================
# Ranking das categorias por receita
# ============================================================

window_categoria = Window.orderBy(F.desc("valor_total"))

vendas_categoria = (
    vendas_categoria
    .withColumn(
        "ranking_receita",
        F.row_number().over(window_categoria)
    )
)

display(
    vendas_categoria.select(
        "ranking_receita",
        "categoria",
        "pedidos",
        "itens_vendidos",
        "valor_produtos",
        "valor_frete",
        "valor_total",
        "ticket_medio_produtos",
        "preco_medio_item",
        "frete_medio",
        "participacao_receita_pct"
    )
    .orderBy("ranking_receita")
)

In [ ]:
# ============================================================
# Validação da agregação por categoria
# ============================================================

validacao_categoria = (
    vendas_categoria
    .agg(
        F.count("*").alias("categorias"),
        
        F.countDistinct("categoria").alias("categorias_distintas"),
        
        F.round(
            F.sum("participacao_receita_pct"),
            2
        ).alias("participacao_total_pct"),
        
        F.round(
            F.sum("valor_total"),
            2
        ).alias("receita_total")
    )
)

display(validacao_categoria)

In [ ]:
# ============================================================
# Inspeção da dimensão de clientes
# ============================================================

dim_customer.printSchema()

In [ ]:
display(
    dim_customer.limit(10)
)

In [ ]:
# ============================================================
# Inspeção da dimensão de pedidos para análise de clientes
# ============================================================

dim_order.printSchema()

In [ ]:
dim_order = spark.table(
    "datalake_mvp.mvp_gold.dim_order"
)

In [ ]:
# ============================================================
# Análise inicial de clientes e recorrência
# ============================================================

base_clientes_pedidos = (
    dim_order.alias("o")
    .join(
        dim_customer
        .select(
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        )
        .alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.order_purchase_timestamp"),
        F.col("c.customer_id"),
        F.col("c.customer_unique_id"),
        F.col("c.customer_city"),
        F.col("c.customer_state")
    )
)

resumo_clientes = (
    base_clientes_pedidos
    .agg(
        F.countDistinct("customer_id").alias("customer_ids"),
        F.countDistinct("customer_unique_id").alias("clientes_unicos"),
        F.countDistinct("order_id").alias("pedidos")
    )
)

display(resumo_clientes)

In [ ]:
# ============================================================
# Análise inicial de clientes e recorrência
# ============================================================

base_clientes_pedidos = (
    dim_order.alias("o")
    .join(
        dim_customer
        .select(
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        )
        .alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.order_purchase_timestamp"),
        F.col("c.customer_id"),
        F.col("c.customer_unique_id"),
        F.col("c.customer_city"),
        F.col("c.customer_state")
    )
)

resumo_clientes = (
    base_clientes_pedidos
    .agg(
        F.countDistinct("customer_id").alias("customer_ids"),
        F.countDistinct("customer_unique_id").alias("clientes_unicos"),
        F.countDistinct("order_id").alias("pedidos")
    )
)

display(resumo_clientes)

In [ ]:
# ============================================================
# Frequência de compra por cliente único
# ============================================================

frequencia_clientes = (
    base_clientes_pedidos
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("quantidade_pedidos"),
        F.min("order_purchase_timestamp").alias("primeira_compra"),
        F.max("order_purchase_timestamp").alias("ultima_compra")
    )
)

resumo_recorrencia = (
    frequencia_clientes
    .agg(
        F.count("*").alias("clientes_unicos"),

        F.sum(
            F.when(F.col("quantidade_pedidos") == 1, 1).otherwise(0)
        ).alias("clientes_uma_compra"),

        F.sum(
            F.when(F.col("quantidade_pedidos") > 1, 1).otherwise(0)
        ).alias("clientes_recorrentes"),

        F.max("quantidade_pedidos").alias("maior_numero_pedidos")
    )
    .withColumn(
        "taxa_recorrencia_pct",
        F.round(
            F.col("clientes_recorrentes")
            / F.col("clientes_unicos") * 100,
            2
        )
    )
)

display(resumo_recorrencia)

In [ ]:
# ============================================================
# Agregação financeira no nível do pedido
# ============================================================

receita_por_pedido = (
    fact_order_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("quantidade_itens"),

        F.sum("price").alias("valor_produtos"),

        F.sum("freight_value").alias("valor_frete"),

        F.sum("item_total_value").alias("valor_pedido")
    )
)

# ============================================================
# Validação da granularidade
# ============================================================

validacao_receita_pedido = (
    receita_por_pedido
    .agg(
        F.count("*").alias("pedidos"),
        F.countDistinct("order_id").alias("pedidos_distintos"),
        F.sum("quantidade_itens").alias("itens"),
        F.round(F.sum("valor_produtos"), 2).alias("valor_produtos"),
        F.round(F.sum("valor_frete"), 2).alias("valor_frete"),
        F.round(F.sum("valor_pedido"), 2).alias("receita_total")
    )
)

display(validacao_receita_pedido)

In [ ]:
# ============================================================
# Base financeira por cliente
# Granularidade: 1 linha por pedido
# ============================================================

base_cliente_financeira = (
    base_clientes_pedidos.alias("c")
    .join(
        receita_por_pedido.alias("r"),
        F.col("c.order_id") == F.col("r.order_id"),
        "left"
    )
    .select(
        F.col("c.customer_unique_id"),
        F.col("c.order_id"),
        F.col("c.order_purchase_timestamp"),
        F.col("r.quantidade_itens"),
        F.col("r.valor_produtos"),
        F.col("r.valor_frete"),
        F.col("r.valor_pedido")
    )
)

display(
    base_cliente_financeira.agg(
        F.countDistinct("order_id").alias("pedidos"),
        F.countDistinct("customer_unique_id").alias("clientes_unicos"),
        F.sum(
            F.when(F.col("valor_pedido").isNull(), 1).otherwise(0)
        ).alias("pedidos_sem_valor"),
        F.round(F.sum("valor_pedido"), 2).alias("receita_total")
    )
)

In [ ]:
# ============================================================
# Agregação analítica por cliente único
# Granularidade: 1 linha por customer_unique_id
# ============================================================

agg_clientes = (
    base_cliente_financeira
    .groupBy("customer_unique_id")
    .agg(
        # Comportamento de compra
        F.countDistinct("order_id").alias("quantidade_pedidos"),

        F.sum(
            F.coalesce(F.col("quantidade_itens"), F.lit(0))
        ).alias("quantidade_itens"),

        # Financeiro
        F.sum("valor_produtos").alias("valor_produtos"),
        F.sum("valor_frete").alias("valor_frete"),
        F.sum("valor_pedido").alias("receita_total"),

        # Ciclo do cliente
        F.min("order_purchase_timestamp").alias("primeira_compra"),
        F.max("order_purchase_timestamp").alias("ultima_compra"),

        # Pedidos efetivamente monetizados
        F.count("valor_pedido").alias("pedidos_com_valor")
    )

    # Ticket médio apenas sobre pedidos com valor
    .withColumn(
        "ticket_medio",
        F.when(
            F.col("pedidos_com_valor") > 0,
            F.round(
                F.col("receita_total") / F.col("pedidos_com_valor"),
                2
            )
        )
    )

    # Recorrência
    .withColumn(
        "flag_recorrente",
        F.when(
            F.col("quantidade_pedidos") > 1,
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    # Tempo entre primeira e última compra
    .withColumn(
        "dias_entre_primeira_ultima_compra",
        F.datediff(
            F.to_date("ultima_compra"),
            F.to_date("primeira_compra")
        )
    )
)

In [ ]:
# ============================================================
# Validação da agg_clientes
# ============================================================

validacao_agg_clientes = (
    agg_clientes
    .agg(
        F.count("*").alias("clientes"),
        F.countDistinct("customer_unique_id").alias("clientes_distintos"),

        F.sum("quantidade_pedidos").alias("pedidos"),
        F.sum("quantidade_itens").alias("itens"),

        F.sum("flag_recorrente").alias("clientes_recorrentes"),

        F.round(
            F.sum("receita_total"), 2
        ).alias("receita_total"),

        F.round(
            F.avg("ticket_medio"), 2
        ).alias("ticket_medio_medio")
    )
)

display(validacao_agg_clientes)

In [ ]:
# ============================================================
# Comparativo: clientes de compra única x recorrentes
# ============================================================

comparativo_recorrencia = (
    agg_clientes
    .withColumn(
        "perfil_cliente",
        F.when(
            F.col("flag_recorrente") == 1,
            F.lit("Recorrente")
        ).otherwise(F.lit("Compra única"))
    )
    .groupBy("perfil_cliente")
    .agg(
        F.count("*").alias("clientes"),

        F.sum("quantidade_pedidos").alias("pedidos"),

        F.sum("quantidade_itens").alias("itens"),

        F.round(
            F.sum("receita_total"), 2
        ).alias("receita_total"),

        F.round(
            F.avg("quantidade_pedidos"), 2
        ).alias("pedidos_medio_cliente"),

        F.round(
            F.avg("quantidade_itens"), 2
        ).alias("itens_medio_cliente"),

        F.round(
            F.avg("ticket_medio"), 2
        ).alias("ticket_medio_cliente")
    )
)

# Participação sobre o total
total_clientes = agg_clientes.count()

total_receita = (
    agg_clientes
    .agg(F.sum("receita_total").alias("total"))
    .first()["total"]
)

comparativo_recorrencia = (
    comparativo_recorrencia
    .withColumn(
        "participacao_clientes_pct",
        F.round(
            F.col("clientes") / F.lit(total_clientes) * 100,
            2
        )
    )
    .withColumn(
        "participacao_receita_pct",
        F.round(
            F.col("receita_total") / F.lit(total_receita) * 100,
            2
        )
    )
    .orderBy(F.desc("clientes"))
)

display(comparativo_recorrencia)

In [ ]:
# ============================================================
# Complemento do comparativo de recorrência
# ============================================================

comparativo_recorrencia_final = (
    comparativo_recorrencia

    # Receita média gerada por cliente no período
    .withColumn(
        "receita_media_cliente",
        F.round(
            F.col("receita_total") / F.col("clientes"),
            2
        )
    )

    # Ticket médio agregado do grupo
    .withColumn(
        "ticket_medio_pedido",
        F.round(
            F.col("receita_total") / F.col("pedidos"),
            2
        )
    )

    # Itens médios por pedido
    .withColumn(
        "itens_medio_pedido",
        F.round(
            F.col("itens") / F.col("pedidos"),
            2
        )
    )
)

display(comparativo_recorrencia_final)

In [ ]:
# ============================================================
# LOGÍSTICA - Visão geral das entregas
# Granularidade: pedido
# ============================================================

dim_order = spark.table(
    "datalake_mvp.mvp_gold.dim_order"
)

kpis_logistica = (
    dim_order
    .agg(
        # Universo
        F.count("*").alias("total_pedidos"),

        # Pedidos com entrega efetivamente registrada
        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNotNull(),
                1
            ).otherwise(0)
        ).alias("pedidos_entregues"),

        # Pedidos sem data de entrega
        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNull(),
                1
            ).otherwise(0)
        ).alias("pedidos_sem_entrega"),

        # Atrasos
        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1,
                1
            ).otherwise(0)
        ).alias("pedidos_atrasados"),

        # Tempo médio real de entrega
        F.round(
            F.avg("delivery_time_days"),
            2
        ).alias("tempo_medio_entrega_dias"),

        # Mediana
        F.expr(
            "percentile_approx(delivery_time_days, 0.5)"
        ).alias("mediana_entrega_dias"),

        # P90
        F.expr(
            "percentile_approx(delivery_time_days, 0.9)"
        ).alias("p90_entrega_dias"),

        # Maior tempo observado
        F.max(
            "delivery_time_days"
        ).alias("maior_tempo_entrega_dias")
    )
)

display(kpis_logistica)

In [ ]:
# ============================================================
# Taxas logísticas
# ============================================================

kpis_logistica_taxas = (
    kpis_logistica
    .withColumn(
        "taxa_entrega_pct",
        F.round(
            F.col("pedidos_entregues")
            / F.col("total_pedidos") * 100,
            2
        )
    )
    .withColumn(
        "taxa_atraso_sobre_total_pct",
        F.round(
            F.col("pedidos_atrasados")
            / F.col("total_pedidos") * 100,
            2
        )
    )
    .withColumn(
        "taxa_atraso_sobre_entregues_pct",
        F.round(
            F.col("pedidos_atrasados")
            / F.col("pedidos_entregues") * 100,
            2
        )
    )
)

display(kpis_logistica_taxas)

In [ ]:
# ============================================================
# LOGÍSTICA - Base de pedidos por localização do cliente
# ============================================================

base_logistica_uf = (
    dim_order.alias("o")
    .join(
        dim_customer
        .select(
            "customer_id",
            "customer_state"
        )
        .alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .select(
        F.col("o.order_id"),
        F.col("c.customer_state").alias("uf"),
        F.col("o.order_delivered_customer_date"),
        F.col("o.delivery_time_days"),
        F.col("o.delivery_delay_days"),
        F.col("o.flag_late_delivery")
    )
)

In [ ]:
# ============================================================
# Desempenho logístico por UF
# ============================================================

logistica_uf = (
    base_logistica_uf
    .groupBy("uf")
    .agg(
        F.countDistinct("order_id").alias("pedidos"),

        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNotNull(),
                1
            ).otherwise(0)
        ).alias("pedidos_entregues"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1,
                1
            ).otherwise(0)
        ).alias("pedidos_atrasados"),

        F.round(
            F.avg("delivery_time_days"),
            2
        ).alias("tempo_medio_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.5)"
        ).alias("mediana_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.9)"
        ).alias("p90_entrega_dias"),

        F.round(
            F.avg(
                F.when(
                    F.col("delivery_delay_days") > 0,
                    F.col("delivery_delay_days")
                )
            ),
            2
        ).alias("atraso_medio_quando_atrasado_dias")
    )
    .withColumn(
        "taxa_atraso_pct",
        F.round(
            F.col("pedidos_atrasados")
            / F.col("pedidos_entregues") * 100,
            2
        )
    )
    .orderBy(F.desc("taxa_atraso_pct"))
)

display(logistica_uf)

In [ ]:
# ============================================================
# LOGÍSTICA - Priorização operacional por UF
# ============================================================

taxa_global_atraso = 6.77

volume_medio_uf = (
    logistica_uf
    .agg(
        F.avg("pedidos_entregues").alias("volume_medio")
    )
    .first()["volume_medio"]
)

print(f"Taxa global de atraso: {taxa_global_atraso:.2f}%")
print(f"Volume médio de entregas por UF: {volume_medio_uf:.2f}")

In [ ]:
# ============================================================
# Matriz volume x taxa de atraso
# ============================================================

logistica_uf_priorizacao = (
    logistica_uf

    .withColumn(
        "perfil_operacional",
        F.when(
            (F.col("taxa_atraso_pct") > taxa_global_atraso) &
            (F.col("pedidos_entregues") >= volume_medio_uf),
            "Alta taxa | Alto volume"
        )
        .when(
            (F.col("taxa_atraso_pct") > taxa_global_atraso) &
            (F.col("pedidos_entregues") < volume_medio_uf),
            "Alta taxa | Baixo volume"
        )
        .when(
            (F.col("taxa_atraso_pct") <= taxa_global_atraso) &
            (F.col("pedidos_entregues") >= volume_medio_uf),
            "Baixa taxa | Alto volume"
        )
        .otherwise(
            "Baixa taxa | Baixo volume"
        )
    )

    # Participação nos atrasos totais
    .withColumn(
        "participacao_atrasos_pct",
        F.round(
            F.col("pedidos_atrasados") / F.lit(6535) * 100,
            2
        )
    )

    .orderBy(
        F.desc("pedidos_atrasados")
    )
)

display(logistica_uf_priorizacao)

In [ ]:
# ============================================================
# LOGÍSTICA - Base seller x pedido
# ============================================================

base_seller_pedido = (
    fact_order_items
    .select(
        "order_id",
        "seller_id"
    )
    .distinct()

    .join(
        dim_order
        .select(
            "order_id",
            "delivery_time_days",
            "delivery_delay_days",
            "flag_late_delivery",
            "order_delivered_customer_date"
        ),
        "order_id",
        "inner"
    )
)

print(f"Relações seller x pedido: {base_seller_pedido.count():,}")
print(
    f"Sellers distintos: "
    f"{base_seller_pedido.select('seller_id').distinct().count():,}"
)

In [ ]:
# ============================================================
# Desempenho logístico por seller
# ============================================================

logistica_seller = (
    base_seller_pedido
    .groupBy("seller_id")
    .agg(
        F.countDistinct("order_id").alias("pedidos"),

        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNotNull(),
                1
            ).otherwise(0)
        ).alias("pedidos_entregues"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1,
                1
            ).otherwise(0)
        ).alias("pedidos_atrasados"),

        F.round(
            F.avg("delivery_time_days"), 2
        ).alias("tempo_medio_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.9)"
        ).alias("p90_entrega_dias"),

        F.round(
            F.avg(
                F.when(
                    F.col("delivery_delay_days") > 0,
                    F.col("delivery_delay_days")
                )
            ), 2
        ).alias("atraso_medio_quando_atrasado_dias")
    )

    .withColumn(
        "taxa_atraso_pct",
        F.when(
            F.col("pedidos_entregues") > 0,
            F.round(
                F.col("pedidos_atrasados")
                / F.col("pedidos_entregues") * 100,
                2
            )
        )
    )
)

In [ ]:
# ============================================================
# Distribuição do volume por seller
# ============================================================

distribuicao_sellers = (
    logistica_seller
    .agg(
        F.count("*").alias("sellers"),

        F.round(
            F.avg("pedidos_entregues"), 2
        ).alias("media_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.5)"
        ).alias("mediana_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.75)"
        ).alias("p75_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.9)"
        ).alias("p90_entregas"),

        F.max("pedidos_entregues").alias("max_entregas")
    )
)

display(distribuicao_sellers)

In [ ]:
# ============================================================
# Sellers com volume relevante
# Corte orientado pelos dados: P75 = 21 entregas
# ============================================================

CORTE_VOLUME_SELLER = 21

sellers_relevantes = (
    logistica_seller
    .filter(
        F.col("pedidos_entregues") >= CORTE_VOLUME_SELLER
    )
    .withColumn(
        "acima_taxa_global",
        F.when(
            F.col("taxa_atraso_pct") > 6.77,
            1
        ).otherwise(0)
    )
)

display(
    sellers_relevantes
    .orderBy(
        F.desc("taxa_atraso_pct"),
        F.desc("pedidos_entregues")
    )
    .limit(20)
)

In [ ]:
# ============================================================
# LOGÍSTICA - Seller x UF
# ============================================================

base_seller_uf = (
    fact_order_items
    .select(
        "order_id",
        "seller_id"
    )
    .distinct()

    .join(
        dim_order
        .select(
            "order_id",
            "customer_id",
            "delivery_time_days",
            "delivery_delay_days",
            "flag_late_delivery",
            "order_delivered_customer_date"
        ),
        "order_id",
        "inner"
    )

    .join(
        dim_customer
        .select(
            "customer_id",
            "customer_state"
        ),
        "customer_id",
        "left"
    )
)

seller_uf = (
    base_seller_uf
    .groupBy(
        "seller_id",
        "customer_state"
    )
    .agg(
        F.countDistinct("order_id").alias("pedidos"),

        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNotNull(),
                1
            ).otherwise(0)
        ).alias("pedidos_entregues"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1,
                1
            ).otherwise(0)
        ).alias("pedidos_atrasados"),

        F.round(
            F.avg("delivery_time_days"),
            2
        ).alias("tempo_medio_entrega_dias")
    )

    .withColumn(
        "taxa_atraso_pct",
        F.when(
            F.col("pedidos_entregues") > 0,
            F.round(
                F.col("pedidos_atrasados")
                / F.col("pedidos_entregues") * 100,
                2
            )
        )
    )
)

In [ ]:
# ============================================================
# Sellers com maior impacto nos atrasos do RJ
# ============================================================

seller_rj = (
    seller_uf
    .filter(
        F.col("customer_state") == "RJ"
    )
    .filter(
        F.col("pedidos_entregues") >= 21
    )
    .withColumn(
        "participacao_atrasos_rj_pct",
        F.round(
            F.col("pedidos_atrasados")
            / F.lit(1495) * 100,
            2
        )
    )
    .orderBy(
        F.desc("pedidos_atrasados"),
        F.desc("taxa_atraso_pct")
    )
)

display(
    seller_rj.limit(20)
)

In [ ]:
# ============================================================
# Concentração dos atrasos por seller no RJ
# ============================================================

window_ranking = Window.orderBy(
    F.desc("pedidos_atrasados"),
    F.desc("taxa_atraso_pct")
)

seller_rj_ranking = (
    seller_rj
    .withColumn(
        "ranking_atrasos",
        F.row_number().over(window_ranking)
    )
)

concentracao_rj = (
    seller_rj_ranking
    .agg(
        F.sum(
            F.when(
                F.col("ranking_atrasos") <= 5,
                F.col("pedidos_atrasados")
            ).otherwise(0)
        ).alias("atrasos_top5"),

        F.sum(
            F.when(
                F.col("ranking_atrasos") <= 10,
                F.col("pedidos_atrasados")
            ).otherwise(0)
        ).alias("atrasos_top10"),

        F.sum(
            F.when(
                F.col("ranking_atrasos") <= 20,
                F.col("pedidos_atrasados")
            ).otherwise(0)
        ).alias("atrasos_top20")
    )
    .withColumn(
        "participacao_top5_pct",
        F.round(F.col("atrasos_top5") / F.lit(1495) * 100, 2)
    )
    .withColumn(
        "participacao_top10_pct",
        F.round(F.col("atrasos_top10") / F.lit(1495) * 100, 2)
    )
    .withColumn(
        "participacao_top20_pct",
        F.round(F.col("atrasos_top20") / F.lit(1495) * 100, 2)
    )
)

display(concentracao_rj)

In [ ]:
# ============================================================
# Parâmetros derivados dos dados
# ============================================================

metricas_globais = (
    dim_order
    .agg(
        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNotNull(), 1
            ).otherwise(0)
        ).alias("pedidos_entregues"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1, 1
            ).otherwise(0)
        ).alias("pedidos_atrasados")
    )
    .first()
)

TOTAL_ENTREGUES = metricas_globais["pedidos_entregues"]
TOTAL_ATRASADOS = metricas_globais["pedidos_atrasados"]

TAXA_GLOBAL_ATRASO = round(
    TOTAL_ATRASADOS / TOTAL_ENTREGUES * 100,
    2
)

TOTAL_ATRASOS_RJ = (
    logistica_uf
    .filter(F.col("uf") == "RJ")
    .select("pedidos_atrasados")
    .first()["pedidos_atrasados"]
)

print(f"Entregues: {TOTAL_ENTREGUES:,}")
print(f"Atrasados: {TOTAL_ATRASADOS:,}")
print(f"Taxa global: {TAXA_GLOBAL_ATRASO}%")
print(f"Atrasos RJ: {TOTAL_ATRASOS_RJ:,}")

In [ ]:
# ============================================================
# LOGÍSTICA - Base pedido x categoria
# ============================================================

base_categoria_pedido = (
    fact_order_items.alias("f")
    .join(
        dim_product
        .select(
            "product_id",
            "product_category_name_english"
        )
        .alias("p"),
        F.col("f.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("f.order_id"),
        F.coalesce(
            F.col("p.product_category_name_english"),
            F.lit("unknown")
        ).alias("categoria")
    )
    .distinct()

    .join(
        dim_order
        .select(
            "order_id",
            "order_delivered_customer_date",
            "delivery_time_days",
            "delivery_delay_days",
            "flag_late_delivery"
        ),
        "order_id",
        "inner"
    )
)

print(f"Relações pedido x categoria: {base_categoria_pedido.count():,}")
print(
    "Categorias distintas:",
    base_categoria_pedido.select("categoria").distinct().count()
)

In [ ]:
# ============================================================
# Indicadores logísticos por categoria
# ============================================================

logistica_categoria = (
    base_categoria_pedido
    .groupBy("categoria")
    .agg(
        F.countDistinct("order_id").alias("pedidos"),

        F.sum(
            F.when(
                F.col("order_delivered_customer_date").isNotNull(), 1
            ).otherwise(0)
        ).alias("pedidos_entregues"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1, 1
            ).otherwise(0)
        ).alias("pedidos_atrasados"),

        F.round(
            F.avg("delivery_time_days"), 2
        ).alias("tempo_medio_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.5)"
        ).alias("mediana_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.9)"
        ).alias("p90_entrega_dias"),

        F.round(
            F.avg(
                F.when(
                    F.col("delivery_delay_days") > 0,
                    F.col("delivery_delay_days")
                )
            ), 2
        ).alias("atraso_medio_quando_atrasado_dias")
    )

    .withColumn(
        "taxa_atraso_pct",
        F.when(
            F.col("pedidos_entregues") > 0,
            F.round(
                F.col("pedidos_atrasados")
                / F.col("pedidos_entregues") * 100,
                2
            )
        )
    )
)

In [ ]:
# ============================================================
# Distribuição de volume das categorias
# ============================================================

distribuicao_categorias = (
    logistica_categoria
    .agg(
        F.count("*").alias("categorias"),

        F.round(
            F.avg("pedidos_entregues"), 2
        ).alias("media_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.5)"
        ).alias("mediana_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.25)"
        ).alias("p25_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.75)"
        ).alias("p75_entregas"),

        F.expr(
            "percentile_approx(pedidos_entregues, 0.9)"
        ).alias("p90_entregas"),

        F.max("pedidos_entregues").alias("max_entregas")
    )
)

display(distribuicao_categorias)

In [ ]:
# ============================================================
# LOGÍSTICA - Categorias com volume relevante
# Corte: mediana = 242 entregas
# ============================================================

CORTE_VOLUME_CATEGORIA = 242

categorias_relevantes = (
    logistica_categoria
    .filter(
        F.col("pedidos_entregues") >= CORTE_VOLUME_CATEGORIA
    )
    .withColumn(
        "diferenca_taxa_global_pp",
        F.round(
            F.col("taxa_atraso_pct") - F.lit(TAXA_GLOBAL_ATRASO),
            2
        )
    )
    .withColumn(
        "acima_taxa_global",
        F.when(
            F.col("taxa_atraso_pct") > TAXA_GLOBAL_ATRASO,
            1
        ).otherwise(0)
    )
)

display(
    categorias_relevantes
    .orderBy(
        F.desc("taxa_atraso_pct"),
        F.desc("pedidos_entregues")
    )
)

In [ ]:
# ============================================================
# Categorias por impacto absoluto
# ============================================================

categorias_impacto = (
    categorias_relevantes
    .withColumn(
        "participacao_atrasos_pct",
        F.round(
            F.col("pedidos_atrasados")
            / F.lit(TOTAL_ATRASADOS) * 100,
            2
        )
    )
    .orderBy(
        F.desc("pedidos_atrasados"),
        F.desc("taxa_atraso_pct")
    )
)

display(categorias_impacto)

In [ ]:
# ============================================================
# LOGÍSTICA - Características físicas dos produtos
# ============================================================

base_fisica_logistica = (
    fact_order_items.alias("f")
    .join(
        dim_product
        .select(
            "product_id",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        )
        .alias("p"),
        F.col("f.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("f.order_id"),
        F.col("f.product_id"),
        F.col("f.order_item_id"),
        F.col("f.flag_late_delivery"),
        F.col("f.delivery_time_days"),
        F.col("f.delivery_delay_days"),
        F.col("p.product_weight_g"),
        F.col("p.product_length_cm"),
        F.col("p.product_height_cm"),
        F.col("p.product_width_cm")
    )
    .withColumn(
        "volume_cm3",
        F.col("product_length_cm")
        * F.col("product_height_cm")
        * F.col("product_width_cm")
    )
)

display(
    base_fisica_logistica
    .select(
        "product_weight_g",
        "volume_cm3"
    )
    .summary()
)

In [ ]:
# ============================================================
# Distribuição de peso e volume
# ============================================================

distribuicao_fisica = (
    base_fisica_logistica
    .agg(
        F.count("*").alias("itens"),

        F.sum(
            F.when(F.col("product_weight_g").isNull(), 1).otherwise(0)
        ).alias("peso_nulo"),

        F.sum(
            F.when(F.col("volume_cm3").isNull(), 1).otherwise(0)
        ).alias("volume_nulo"),

        F.expr(
            "percentile_approx(product_weight_g, 0.25)"
        ).alias("peso_p25_g"),

        F.expr(
            "percentile_approx(product_weight_g, 0.5)"
        ).alias("peso_mediana_g"),

        F.expr(
            "percentile_approx(product_weight_g, 0.75)"
        ).alias("peso_p75_g"),

        F.expr(
            "percentile_approx(product_weight_g, 0.9)"
        ).alias("peso_p90_g"),

        F.expr(
            "percentile_approx(volume_cm3, 0.25)"
        ).alias("volume_p25_cm3"),

        F.expr(
            "percentile_approx(volume_cm3, 0.5)"
        ).alias("volume_mediana_cm3"),

        F.expr(
            "percentile_approx(volume_cm3, 0.75)"
        ).alias("volume_p75_cm3"),

        F.expr(
            "percentile_approx(volume_cm3, 0.9)"
        ).alias("volume_p90_cm3")
    )
)

display(distribuicao_fisica)

In [ ]:
# ============================================================
# LOGÍSTICA - Faixas de peso
# Baseadas nos quartis da distribuição
# ============================================================

base_peso = (
    base_fisica_logistica
    .filter(F.col("product_weight_g").isNotNull())
    .withColumn(
        "faixa_peso",
        F.when(
            F.col("product_weight_g") <= 300,
            "Q1 | até 300g"
        )
        .when(
            F.col("product_weight_g") <= 700,
            "Q2 | 301g a 700g"
        )
        .when(
            F.col("product_weight_g") <= 1800,
            "Q3 | 701g a 1800g"
        )
        .otherwise(
            "Q4 | acima de 1800g"
        )
    )
)

In [ ]:
logistica_peso = (
    base_peso
    .groupBy("faixa_peso")
    .agg(
        F.count("*").alias("itens"),

        F.countDistinct("order_id").alias("pedidos"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1, 1
            ).otherwise(0)
        ).alias("itens_em_pedidos_atrasados"),

        F.round(
            F.avg("delivery_time_days"), 2
        ).alias("tempo_medio_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.5)"
        ).alias("mediana_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.9)"
        ).alias("p90_entrega_dias"),

        F.round(
            F.avg(
                F.when(
                    F.col("delivery_delay_days") > 0,
                    F.col("delivery_delay_days")
                )
            ), 2
        ).alias("atraso_medio_quando_atrasado_dias")
    )

    .withColumn(
        "taxa_itens_pedidos_atrasados_pct",
        F.round(
            F.col("itens_em_pedidos_atrasados")
            / F.col("itens") * 100,
            2
        )
    )

    .withColumn(
        "ordem_faixa",
        F.when(F.col("faixa_peso").startswith("Q1"), 1)
         .when(F.col("faixa_peso").startswith("Q2"), 2)
         .when(F.col("faixa_peso").startswith("Q3"), 3)
         .otherwise(4)
    )

    .orderBy("ordem_faixa")
)

display(logistica_peso)

In [ ]:
# ============================================================
# LOGÍSTICA - Faixas de volume
# ============================================================

base_volume = (
    base_fisica_logistica
    .filter(F.col("volume_cm3").isNotNull())
    .withColumn(
        "faixa_volume",
        F.when(
            F.col("volume_cm3") <= 2852,
            "Q1 | até 2.852 cm³"
        )
        .when(
            F.col("volume_cm3") <= 6480,
            "Q2 | 2.853 a 6.480 cm³"
        )
        .when(
            F.col("volume_cm3") <= 18375,
            "Q3 | 6.481 a 18.375 cm³"
        )
        .otherwise(
            "Q4 | acima de 18.375 cm³"
        )
    )
)

logistica_volume = (
    base_volume
    .groupBy("faixa_volume")
    .agg(
        F.count("*").alias("itens"),

        F.countDistinct("order_id").alias("pedidos"),

        F.sum(
            F.when(
                F.col("flag_late_delivery") == 1, 1
            ).otherwise(0)
        ).alias("itens_em_pedidos_atrasados"),

        F.round(
            F.avg("delivery_time_days"), 2
        ).alias("tempo_medio_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.5)"
        ).alias("mediana_entrega_dias"),

        F.expr(
            "percentile_approx(delivery_time_days, 0.9)"
        ).alias("p90_entrega_dias")
    )

    .withColumn(
        "taxa_itens_pedidos_atrasados_pct",
        F.round(
            F.col("itens_em_pedidos_atrasados")
            / F.col("itens") * 100,
            2
        )
    )

    .withColumn(
        "ordem_faixa",
        F.when(F.col("faixa_volume").startswith("Q1"), 1)
         .when(F.col("faixa_volume").startswith("Q2"), 2)
         .when(F.col("faixa_volume").startswith("Q3"), 3)
         .otherwise(4)
    )

    .orderBy("ordem_faixa")
)

display(logistica_volume)

#Características físicas e desempenho logístico: 
A análise por quartis identificou uma associação descritiva entre o peso dos produtos e os indicadores de entrega. A proporção de itens pertencentes a pedidos atrasados aumentou de 5,93% no primeiro quartil de peso para 7,16% no quarto quartil, enquanto o tempo médio de entrega passou de 11,50 para 13,49 dias. Para volume, não foi observada progressão consistente entre os três primeiros quartis, porém os produtos do maior quartil apresentaram simultaneamente maior proporção de itens em pedidos atrasados (7,04%) e maior tempo médio de entrega (13,49 dias). Os resultados indicam associação, mas não permitem estabelecer relação causal.